## Homework 1

**Exercise 1**. Create a function that finds and returns all the proper names in the text. It is guaranteed that no sentence in the text begins with a proper name.

In [5]:
import string

def proper_names(text):
  words = text.split()
  names = []

  # start from 1: the first word of the text always begins a sentence
  for i in range(1, len(words)):
    previous = words[i - 1]
    last = previous[len(previous) - 1]     # the last character of the previous word
    if last == '.' or last == '!' or last == '?':
      continue                             # the previous word ended a sentence, so skip this word

    word = words[i]
    if word.startswith("I'"):              # for e.g. I'm, I'll
      continue
    word = word.replace("'s", '')          # For e.g. changes Anna's -> Anna
    for symbol in string.punctuation:      # remove punctuation marks
      word = word.replace(symbol, '')
    if word == 'I':                        # the pronoun I is not a name
      continue

    if word != '' and word[0].isupper():   # a capital letter inside a sentence
      names.append(word)

  return names

Testing the function

In [ ]:
Yesterday Maria and Ilya walked to the market in Moscow. They bought bread and cheese. Then Maria's brother called from Lagos. He said that Ilya should visit Nigeria next year!text = ""
print(proper_names(text))

['Maria', 'Ilya', 'Moscow', 'Maria', 'Lagos', 'Ilya', 'Nigeria']


In [8]:
text = "Last summer Anton flew to India. He loved it! I think Anton's trip to India was great, and I'm sure Anton will go back with Masha."
print(proper_names(text))

['Anton', 'India', 'Anton', 'India', 'Anton', 'Masha']


<br>

**Exercise 2**. Using the function from the previous exercise, make a program that creates a dictionary of frequencies of proper names in the given text (the text is entered via keyboard input). It is forbidden to use any libraries other than `string`!

In [14]:
text = input()
names = proper_names(text)     # the function from Exercise 1

name_freq = {}                 # an empty dictionary: name -> how many times it appears
for name in names:
  if name in name_freq:
    name_freq[name] += 1       # we have seen this name before: add 1
  else:
    name_freq[name] = 1        # the first time we see this name

print(name_freq)

{'Maria': 2, 'Ilya': 2, 'Moscow': 1, 'Lagos': 1, 'Nigeria': 1}


<br>

**Exercise 3**. Explore the `collections` library. Solve the task from *Ex. 2* using this library.

In [17]:
from collections import Counter   

text = input()
names = proper_names(text)         # the function from Exercise 1

name_freq = Counter(names)         # counts every name
print(name_freq)

Counter({'Maria': 2, 'Ilya': 2, 'Moscow': 1, 'Lagos': 1, 'Nigeria': 1})


Exploring the collection

In [18]:
print(dict(name_freq))             # a Counter can be turned into a normal dictionary
print(name_freq['Anna'])           # look up one name, lke we do with dictionary
print(name_freq.most_common(2))    # prints the 2 most frequent names

{'Maria': 2, 'Ilya': 2, 'Moscow': 1, 'Lagos': 1, 'Nigeria': 1}
0
[('Maria', 2), ('Ilya', 2)]


<br>

**Exercise 4**. Print all words including the substring 'play' from the given text (the text is entered via keyboard input).

In [19]:
import string

text = input()
words = text.split()

for word in words:
  for symbol in string.punctuation:     # remove punctuation marks
    word = word.replace(symbol, '')
  if 'play' in word.lower():            # lower() will make "Playing" match too
    print(word)

play
players
playground
Replaying
Playing
display
airplay


I tested with the following texts: They play football and the players enjoyed the playground. Replaying it was fun. Playing, display and airplay!

In my opinion, the program will be partly useful for looking for derivational relatives. My reasoning being that It finds real relatives like *play, player, replaying, playground*, so it works as a quick first pass search but it only compares letters, not meaning. To find derivational relatives properly, a program has to know about word structure, e.g, prefixes, suffixes, stems, etc.

<br>

**Exercise 5.** Suppose we treat in the research all short words as [stop words](https://en.wikipedia.org/wiki/Stop_word?ysclid=miq2cgo6yo878404715). Write a function that removes all short words (1-2 characters in length) using regular expressions.

In [20]:
import re

def remove_short_words(text):
  text = re.sub(r'\b[a-zA-Z]\b', '', text)         # words of 1 letter
  text = re.sub(r'\b[a-zA-Z]{2}\b', '', text)      # words of 2 letters
  text = re.sub(r'\s+', ' ', text)                 # the removed words leave double spaces: make them single
  return text.strip()                              # remove spaces at the start and the end

Testing it

In [21]:
text = "It is a nice day, and we go to the sea. He is at home today."
print(remove_short_words(text))

nice day, and the sea. home today.


<br>

**Exercise 6.** Explore the `frozenset` data type. Think up a task in which frozensets would be the most appropriate data type (the assessment will also take into account the originality of the task). Prove that other data types would be less appropriate. Solve your task.

**The task.** Turkish has vowel harmony: the vowels of a word usually all come from one group, front (`e i ö ü`) or back (`a ı o u`). I check each word for harmony, and I also group the words by their set of vowels (the "vowel signature").

**Why frozenset.** The signature has to ignore the order and the repeats of vowels: `kızlar` (ı, a) and `kapı` (a, ı) have the same signature. So it has to be a set. I also need it as a dictionary key to group the words, and keys must be immutable, so it has to be a frozenset. The front and back groups are fixed, so an immutable type suits them too.

**Why other types are less appropriate.** A `set` or a `list` cannot be a dictionary key. A `tuple` can, but order and repeats matter, so words with the same vowels land in different groups. The last cell shows it: 11 groups with tuples instead of 9.

In [22]:
FRONT = frozenset('eiöü')       # front vowels: a fixed group, so a frozenset
BACK = frozenset('aıou')        # back vowels
VOWELS = FRONT | BACK           # union of the two groups

words = ['ev', 'evler', 'sevgi', 'güzel', 'kızlar', 'kapı', 'okul', 'araba', 'kitap', 'kalem', 'anne', 'televizyon']

groups = {}                     # vowel signature -> words with this signature
for word in words:
  signature = frozenset(word) & VOWELS      # the set of vowels in the word

  if len(signature - FRONT) == 0:           # no vowel outside the front group
    kind = 'front'
  elif len(signature - BACK) == 0:          # no vowel outside the back group
    kind = 'back'
  else:
    kind = 'mixed'
  print(word, kind)

  if signature in groups:
    groups[signature].append(word)
  else:
    groups[signature] = [word]

print()
for signature, group in groups.items():
  print(sorted(signature), group)           # sorted() only to print the vowels in a stable order

ev front
evler front
sevgi front
güzel front
kızlar back
kapı back
okul back
araba back
kitap mixed
kalem mixed
anne mixed
televizyon mixed

['e'] ['ev', 'evler']
['e', 'i'] ['sevgi']
['e', 'ü'] ['güzel']
['a', 'ı'] ['kızlar', 'kapı']
['o', 'u'] ['okul']
['a'] ['araba']
['a', 'i'] ['kitap']
['a', 'e'] ['kalem', 'anne']
['e', 'i', 'o'] ['televizyon']


The proof

In [23]:
# the same grouping with tuples as keys
tuple_groups = {}
for word in words:
  vowels_in_word = []
  for letter in word:
    if letter in VOWELS:
      vowels_in_word.append(letter)
  signature = tuple(vowels_in_word)         # in a tuple the order and the repeats matter

  if signature in tuple_groups:
    tuple_groups[signature].append(word)
  else:
    tuple_groups[signature] = [word]

print(len(tuple_groups), 'groups with tuples, but', len(groups), 'with frozensets')
print(tuple_groups[('ı', 'a')], tuple_groups[('a', 'ı')])    # kızlar and kapı are separated

11 groups with tuples, but 9 with frozensets
['kızlar'] ['kapı']


<br>

**Bonus task**:

Think about how to solve **Ex. 1** (to find all proper nouns in a text), considering that they may also appear at the beginning of a sentence. Propose your solution.

To get the maximum score for this task, do not use any libraries in your solution. You may use libraries to receive partial score.

In [29]:
# My solution: Since a capital letter at the start of a sentence won't be of much help, we use the rest of the text as evidence.
# (1) A capital word in the middle of a sentence is a name.
# (2) A sentence-initial word that also appears in low case somewhere (the, then) is a common word.
# (3) A sentence-initial word that appears in capital in the middle somewhere else is a name.
# (4) With no evidence, we check a short list of common sentence openers; anything else is guessed to be a name.

def proper_names_all(text):
  punctuation = '.,;:!?"()-' + "'"       # my own list: no libraries
  openers = ['the', 'a', 'an', 'he', 'she', 'it', 'they', 'we', 'you', 'this', 'that', 'there', 'then', 'when', 'after', 'before', 'in', 'on', 'at', 'my', 'his', 'her', 'our', 'their', 'if', 'but', 'and', 'so', 'yesterday', 'today', 'tomorrow']

  # step 1: clean every word and remember whether it begins a sentence
  tokens = []
  begins = True                          # the first word of the text begins a sentence
  for raw in text.split():
    word = raw
    if word.startswith("I'"):            # I'm, I'll
      word = 'I'
    word = word.replace("'s", '')        # change Masha's to Masha
    for symbol in punctuation:
      word = word.replace(symbol, '')
    if word != '':
      tokens.append((word, begins))
      last = raw[len(raw) - 1]           # the last character of the original word
      begins = (last == '.' or last == '!' or last == '?')

  # step 2: collect evidence from the whole text
  mid_capital = set()                    # capital in the middle of a sentence
  seen_lower = set()                     # seen in low case anywhere
  for word, begins in tokens:
    if word[0].isupper() and not begins:
      if word != 'I':
        mid_capital.add(word)
    elif word[0].islower():
      seen_lower.add(word)

  # step 3: decide for every word in capital
  names = []
  for word, begins in tokens:
    if not word[0].isupper() or word == 'I':
      continue
    if not begins:
      names.append(word)                 # rule no 1
    elif word.lower() in seen_lower:
      continue                           # rule no 2: a common word
    elif word in mid_capital:
      names.append(word)                 # rule no 3
    elif word.lower() not in openers:
      names.append(word)                 # rule no 4: no evidence, we guess it is a name
  return names

Testing it

In [28]:
text = "Ilya loves Moscow. Paul lives in Lagos. Yesterday Ilya met Paul in Moscow. Then Paul called Masha. Masha said that the weather was nice. Moscow is cold in winter, but Lagos is warm. Winter is long. Mary smiled."
print(proper_names(text))        # the function from Exercise 1, for comparison
print(proper_names_all(text))

['Moscow', 'Lagos', 'Ilya', 'Paul', 'Moscow', 'Paul', 'Masha', 'Lagos']
['Ilya', 'Moscow', 'Paul', 'Lagos', 'Ilya', 'Paul', 'Moscow', 'Paul', 'Masha', 'Masha', 'Moscow', 'Lagos', 'Mary']
